In [1]:
import os, sys
os.chdir('..')
sys.path.insert(0, '.')

from src.ingestion.embedder import search as vector_search
from src.retrieval.hybrid import hybrid_search, bm25_search
from src.retrieval.reranker import rerank

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
def show_results(query, results, score_key, label):
    print(f'\n{"=" * 80}')
    print(f'{label}: {query!r}')
    print("=" * 80)
    for i, r in enumerate(results, 1):
        score = r.get(score_key, 0)
        text = r['text'].replace('\n', ' ')[:120]
        print(f'  {i}. [{r["source"]:25s}] ({score_key}={score:.4f}) {text}...')

In [3]:
queries = [
    'How much does the Business plan cost per month?',
    'Which compliance certifications does Northwind have?',
    'Can I import dashboards from Tableau?',
    'What payment methods do you accept?',
    'Why is my dashboard loading slow?',
]

In [4]:
q = queries[0]
show_results(q, vector_search(q, top_k=5), 'distance', 'Vector search')
show_results(q, bm25_search(q, top_k=5), 'bm25_score', 'BM25 search')
show_results(q, hybrid_search(q, top_k=5), 'score', 'Hybrid (RRF fusion)')
candidates = hybrid_search(q, top_k=20)
show_results(q, rerank(q, candidates, top_k=5), 'rerank_score', 'Hybrid + Rerank')


Vector search: 'How much does the Business plan cost per month?'
  1. [faq.md                   ] (distance=0.4506) ## Pricing and billing  **Q: Can I change plans mid-cycle?** A: Yes. Upgrades are pro-rated to the day. Downgrades take ...
  2. [pricing.md               ] (distance=0.4572) ## Add-ons  - **Additional storage**: $10/month per 100 GB above plan - **Query overage**: $0.0005 per query above plan ...
  3. [migration_guide.md       ] (distance=0.4711) ## Migration assistance  Enterprise customers receive dedicated migration assistance from their assigned engineer. Busin...
  4. [limits_and_quotas.md     ] (distance=0.4768) ## Query limits  | Limit | Free | Starter | Business | Enterprise | |-------|------|---------|----------|------------| |...
  5. [pricing.md               ] (distance=0.5108) ## Plans  ### Free - Up to 3 users - 5 GB data storage - 10K queries per month - Community support only - Public dashboa...

BM25 search: 'How much does the Business plan cost per mon

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Hybrid + Rerank: 'How much does the Business plan cost per month?'
  1. [pricing.md               ] (rerank_score=0.9495) ## Add-ons  - **Additional storage**: $10/month per 100 GB above plan - **Query overage**: $0.0005 per query above plan ...
  2. [limits_and_quotas.md     ] (rerank_score=0.0782) ## Query limits  | Limit | Free | Starter | Business | Enterprise | |-------|------|---------|----------|------------| |...
  3. [pricing.md               ] (rerank_score=-0.4832) ### Business — $299/month per workspace - Up to 50 users - 500 GB data storage - 1M queries per month - Priority email +...
  4. [pricing.md               ] (rerank_score=-1.9441) ## Plans  ### Free - Up to 3 users - 5 GB data storage - 10K queries per month - Community support only - Public dashboa...
  5. [regions_and_sla.md       ] (rerank_score=-2.2972) ## SLA details  ### Uptime targets - **Business plan**: 99.9% monthly (43 minutes max downtime per month) - **Enterprise...


In [5]:
for q in queries[1:]:
    candidates = hybrid_search(q, top_k=20)
    final = rerank(q, candidates, top_k=3)
    print(f'\nQ: {q}')
    for i, r in enumerate(final, 1):
        text = r['text'].replace('\n', ' ')[:120]
        print(f'  {i}. [{r["source"]:25s}] (score={r["rerank_score"]:.3f}) {text}...')


Q: Which compliance certifications does Northwind have?
  1. [security.md              ] (score=7.002) # Security and Compliance  Northwind Cloud treats data security as a first-class concern. This document describes our se...
  2. [policies.md              ] (score=1.355) ## Acceptable Use  What Northwind is not for: - Storing personal medical records (use HIPAA-compliant Enterprise tier if...
  3. [onboarding.md            ] (score=0.301) # Onboarding for Teams  This guide is for teams setting up Northwind Cloud at the organization level. For individual set...

Q: Can I import dashboards from Tableau?
  1. [faq.md                   ] (score=1.022) **Q: How does it compare to Looker / Tableau / Metabase?** A: We focus on the full pipeline (ingestion + warehouse + das...
  2. [migration_guide.md       ] (score=0.376) ## From Tableau  ### What transfers cleanly - SQL queries and data models (export from Tableau, paste into Northwind) - ...
  3. [migration_guide.md       ] (score=0.162)


Q: What payment methods do you accept?
  1. [billing.md               ] (score=4.836) ## Accepted payment methods  | Method | Plans | |--------|-------| | Credit card (Visa, Mastercard, Amex, Discover) | Al...
  2. [billing.md               ] (score=0.431) # Billing and Payments  This document covers how billing works at Northwind Cloud, accepted payment methods, and how to ...
  3. [pricing.md               ] (score=-3.574) ## Billing and invoices  - Monthly billing on the day of signup - Annual billing receives a 20% discount, paid upfront -...



Q: Why is my dashboard loading slow?
  1. [troubleshooting.md       ] (score=5.767) ### "Dashboard loading is slow" - Check **Dashboard → Performance** — shows per-chart load times - Enable **Auto-cache**...
  2. [troubleshooting.md       ] (score=-8.239) ## Query and dashboard issues  ### "Query timed out" - Default timeout is 5 minutes. Increase to 30 minutes in **Setting...
  3. [troubleshooting.md       ] (score=-9.636) ### "Chart shows wrong numbers" - Check that filters from the dashboard level haven't been overridden at the chart level...


In [6]:
from src.retrieval.rag import RAGPipeline

rag = RAGPipeline()
result = rag.query('What is the maximum file upload size on Business plan?')
print('Q:', result['question'])
print('\nA:', result['answer'])
print('\nSources:', ', '.join(result['sources']))

Q: What is the maximum file upload size on Business plan?

A: The maximum single file upload size on the Business plan is 10 GB. You can upload files up to this size using the **Data → Upload** feature and drag-and-drop.

Sources: limits_and_quotas.md, regions_and_sla.md, faq.md, getting_started.md
